In [ ]:
# --- repo-root anchor (added during reorganisation) ---
# Walks up from the working directory to the repo root, so every path below
# resolves whether this notebook is run from code/notebooks/ or the repo root.
from pathlib import Path as _P
REPO_ROOT = _P.cwd().resolve()
while not (REPO_ROOT / 'requirements.txt').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

DATA_DIR       = REPO_ROOT / 'data'
MODELS_DIR     = REPO_ROOT / 'models'
RESULTS_DIR    = REPO_ROOT / 'results'
REFERENCE_DIR  = REPO_ROOT / 'reference'
VALIDATION_DIR = RESULTS_DIR / 'validation'
# Staging area for freshly trained weights. Training ALWAYS writes here,
# never straight into models/, so released checkpoints are never overwritten.
CKPT_DIR       = MODELS_DIR / 'rbc_ckpts'
CKPT_DIR.mkdir(parents=True, exist_ok=True)
VALIDATION_DIR.mkdir(parents=True, exist_ok=True)
print('repo root:', REPO_ROOT)


In [ ]:
import cv2, numpy as np
from cellpose import models

img = cv2.imread(str(RESULTS_DIR / 'AllSubtypes_OneModel' / 'V1' / 'first_frame.png'))  # your saved frame
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

model = models.CellposeModel(gpu=False, pretrained_model="cyto3")  # built-in known-good
masks, flows, styles = model.eval(
    gray, diameter=None, channels=[0,0],  # auto-estimate size
    flow_threshold=0.2, cellprob_threshold=-3.0
)
print("max mask id:", np.max(masks))


In [ ]:
import cv2, numpy as np, time

# 1) Load and inspect size
img = cv2.imread(str(RESULTS_DIR / 'AllSubtypes_OneModel' / 'V1' / 'first_frame.png'))
h, w = img.shape[:2]
print(f"image size: {w}x{h}")

# 2) Downscale to keep the longer side <= 1200 px (tweak as needed)
max_side = 1200
scale = min(1.0, max_side / max(h, w))
if scale < 1.0:
    new_w, new_h = int(w*scale), int(h*scale)
    img_small = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)
else:
    img_small = img.copy()
print(f"resized to: {img_small.shape[1]}x{img_small.shape[0]} (scale={scale:.3f})")

# 3) Choose channel: try red channel first (often best for cyto), else grayscale
use_red = False
if use_red:
    # BGR -> use R as 'cyto' and no nuclear channel
    channels = [2, 0]
    input_img = img_small
else:
    channels = [0, 0]
    input_img = cv2.cvtColor(img_small, cv2.COLOR_BGR2GRAY)

# 4) Model: classic 'cyto3' on CPU (make sure you installed cellpose==2.2.3)
model = models.CellposeModel(gpu=False, pretrained_model="cyto3")
print("Cellpose ready.")

# # 5) Diameter: estimate (set None for auto or provide a number in pixels at RESIZED scale)
# # If a typical cell is ~30 px in the ORIGINAL, use diameter = 30*scale at the resized scale.
# diam_original_px = 30  # <-- adjust if you know it
# diameter = max(8, int(diam_original_px * scale))  # keep >=8px to be safe
# print(f"using diameter ~ {diameter} px at resized scale")

# 6) Run and time it
t0 = time.time()
masks, flows, styles = model.eval(
    input_img,
    diameter=None,            # or None to auto-estimate
    channels=channels,
    flow_threshold=0.2,
    cellprob_threshold=-6.0
)
dt = time.time() - t0
print("max mask id:", int(np.max(masks)), "| time: %.2f s" % dt)

# 7) Optional: visualize overlay
try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(6,6))
    if input_img.ndim == 2:
        plt.imshow(input_img, cmap='gray')
    else:
        plt.imshow(cv2.cvtColor(input_img, cv2.COLOR_BGR2RGB))
    plt.imshow(masks, alpha=0.4)
    plt.axis('off'); plt.show()
except Exception as e:
    print("Visualization skipped:", e)
